# Rossmann Store Sales Forecasting – Complete Machine Learning Notebook

**Project:** Sales Forecasting Across Multiple Retail Stores  
**Goal:** Predict daily store sales using customer behaviour, promotions, holidays, competition, seasonality and store information.

This notebook covers the PDF requirements:
1. Data loading and quality checks
2. Exploratory Data Analysis (EDA)
3. Customer purchasing behaviour analysis
4. Logging
5. Feature engineering and preprocessing
6. Scikit-learn pipelines and machine learning models
7. Loss/evaluation analysis
8. Feature importance and prediction confidence intervals
9. Model serialization
10. Time-series analysis
11. LSTM deep learning model
12. MLflow experiment tracking
13. Test-set prediction generation

In [ ]:
# Run once if packages are missing
# !pip install pandas numpy matplotlib seaborn scikit-learn scipy statsmodels joblib mlflow tensorflow

import os
import sys
import warnings
import logging
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import joblib

from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Project Setup and Logging

In [ ]:
PROJECT_ROOT = Path.cwd()

# Automatic search for the dataset folder.
possible_data_dirs = [
    PROJECT_ROOT / "data" / "raw",
    Path("/mnt/data/Rossmann_Store_Sales_Project/data/raw"),
    PROJECT_ROOT
]

DATA_DIR = next((p for p in possible_data_dirs if (p / "train.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "train.csv not found. Put train.csv, test.csv and store.csv in data/raw/"
    )

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = PROJECT_ROOT / "rossmann_project.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE), logging.StreamHandler()]
)

logger = logging.getLogger("rossmann")
logger.info("Project started")
print("Data directory:", DATA_DIR)
print("Log file:", LOG_FILE)

## 2. Load the Datasets

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", low_memory=False)
test = pd.read_csv(DATA_DIR / "test.csv", low_memory=False)
store = pd.read_csv(DATA_DIR / "store.csv", low_memory=False)
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", low_memory=False)

logger.info("Datasets loaded successfully")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Store shape:", store.shape)
print("Sample submission shape:", sample_submission.shape)

display(train.head())
display(store.head())

## 3. Initial Data Overview and Data Quality

In [ ]:
for name, df in {
    "train": train,
    "test": test,
    "store": store
}.items():
    print(f"\n{'='*20} {name.upper()} {'='*20}")
    print("Shape:", df.shape)
    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))
    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))
    print("\nDuplicate rows:", df.duplicated().sum())

display(train.describe(include="all").T)

### Key observation
`Sales` is the main target. `Customers` is available in training data, but it is **not available in the competition test data**, so it must not be used as a production feature for predicting future test sales.

## 4. Merge Store Information with Train and Test Data

In [ ]:
train_df = train.merge(store, on="Store", how="left")
test_df = test.merge(store, on="Store", how="left")

train_df["Date"] = pd.to_datetime(train_df["Date"])
test_df["Date"] = pd.to_datetime(test_df["Date"])

train_df = train_df.sort_values("Date").reset_index(drop=True)
test_df = test_df.sort_values("Date").reset_index(drop=True)

logger.info("Store information merged with train and test datasets")

print("Merged train shape:", train_df.shape)
print("Merged test shape:", test_df.shape)
display(train_df.head())

## 5. EDA – Target Distribution and Basic Sales Behaviour

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(train_df["Sales"], bins=80, ax=axes[0])
axes[0].set_title("Sales Distribution")

sns.boxplot(x=train_df["Sales"], ax=axes[1])
axes[1].set_title("Sales Boxplot")

plt.tight_layout()
plt.show()

print(train_df["Sales"].describe())

In [ ]:
daily_sales = train_df.groupby("Date", as_index=False)["Sales"].sum()

plt.figure(figsize=(16, 5))
plt.plot(daily_sales["Date"], daily_sales["Sales"])
plt.title("Total Daily Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.show()

## 6. Task 1 – Promotion Distribution in Train vs Test

In [ ]:
promo_compare = pd.DataFrame({
    "Train": train_df["Promo"].value_counts(normalize=True).sort_index(),
    "Test": test_df["Promo"].value_counts(normalize=True).sort_index()
}).fillna(0)

display(promo_compare)

promo_compare.plot(kind="bar", figsize=(8, 5))
plt.title("Promotion Distribution: Train vs Test")
plt.ylabel("Proportion")
plt.show()

## 7. Sales Before, During and After Holidays

In [ ]:
def holiday_period(date_series, holiday_dates):
    result = np.full(len(date_series), "Normal", dtype=object)
    dates = pd.Series(date_series).dt.normalize()

    holiday_set = set(pd.to_datetime(holiday_dates).normalize())
    before_set = set((pd.to_datetime(holiday_dates) - pd.Timedelta(days=1)).normalize())
    after_set = set((pd.to_datetime(holiday_dates) + pd.Timedelta(days=1)).normalize())

    result[dates.isin(before_set)] = "Before Holiday"
    result[dates.isin(holiday_set)] = "Holiday"
    result[dates.isin(after_set)] = "After Holiday"
    return result

holiday_dates = train_df.loc[train_df["StateHoliday"].astype(str) != "0", "Date"].drop_duplicates()
holiday_df = train_df.copy()
holiday_df["HolidayPeriod"] = holiday_period(holiday_df["Date"], holiday_dates)

holiday_summary = holiday_df.groupby("HolidayPeriod")["Sales"].agg(["mean", "median", "count"])
display(holiday_summary)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=holiday_df.groupby("HolidayPeriod", as_index=False)["Sales"].mean(),
    x="HolidayPeriod", y="Sales"
)
plt.title("Average Sales Before, During and After Holidays")
plt.show()

## 8. Seasonal Purchase Behaviour

In [ ]:
seasonal = train_df.copy()
seasonal["Month"] = seasonal["Date"].dt.month
seasonal["Year"] = seasonal["Date"].dt.year

monthly_sales = seasonal.groupby(["Year", "Month"], as_index=False)["Sales"].mean()

plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_sales, x="Month", y="Sales", hue="Year", marker="o")
plt.title("Average Sales by Month and Year")
plt.xticks(range(1, 13))
plt.show()

christmas = seasonal[seasonal["Date"].dt.month == 12]["Sales"].mean()
easter_month = seasonal[seasonal["Date"].dt.month.isin([3, 4])]["Sales"].mean()

print("Average December sales:", round(christmas, 2))
print("Average March-April sales:", round(easter_month, 2))

## 9. Correlation Between Sales and Customers

In [ ]:
corr = train_df[["Sales", "Customers"]].corr()
display(corr)

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Sales vs Customers Correlation")
plt.show()

sample_for_plot = train_df.sample(min(50000, len(train_df)), random_state=RANDOM_STATE)
plt.figure(figsize=(8, 5))
sns.scatterplot(data=sample_for_plot, x="Customers", y="Sales", alpha=0.25)
plt.title("Sales vs Customers")
plt.show()

## 10. How Promotions Affect Sales and Customers

In [ ]:
promo_effect = train_df.groupby("Promo")[["Sales", "Customers"]].mean()
display(promo_effect)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
promo_effect["Sales"].plot(kind="bar", ax=axes[0], title="Average Sales by Promo")
promo_effect["Customers"].plot(kind="bar", ax=axes[1], title="Average Customers by Promo")
plt.tight_layout()
plt.show()

promo_store = train_df.groupby(["Store", "Promo"])["Sales"].mean().unstack()
promo_store["Promo_Lift"] = promo_store.get(1, 0) - promo_store.get(0, 0)

display(promo_store["Promo_Lift"].sort_values(ascending=False).head(10).to_frame("Promo_Lift"))

## 11. Store Open/Closed Behaviour and Weekends

In [ ]:
open_summary = train_df.groupby("Open")[["Sales", "Customers"]].mean()
display(open_summary)

weekend_summary = train_df.assign(
    IsWeekend=train_df["Date"].dt.dayofweek.isin([5, 6]).astype(int)
).groupby("IsWeekend")[["Sales", "Customers"]].mean()

display(weekend_summary)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=train_df.assign(
        DayName=train_df["Date"].dt.day_name()
    ).groupby("DayName", as_index=False)["Sales"].mean(),
    x="DayName", y="Sales",
    order=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
)
plt.xticks(rotation=30)
plt.title("Average Sales by Day of Week")
plt.show()

## 12. Assortment and Competition Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=train_df.groupby("Assortment", as_index=False)["Sales"].mean(),
    x="Assortment", y="Sales", ax=axes[0]
)
axes[0].set_title("Average Sales by Assortment")

competition_sample = train_df.dropna(subset=["CompetitionDistance"]).copy()
competition_sample["CompetitionDistance"] = competition_sample["CompetitionDistance"].clip(
    upper=competition_sample["CompetitionDistance"].quantile(0.99)
)

sns.scatterplot(
    data=competition_sample.sample(min(50000, len(competition_sample)), random_state=RANDOM_STATE),
    x="CompetitionDistance", y="Sales", alpha=0.2, ax=axes[1]
)
axes[1].set_title("Competition Distance vs Sales (99th percentile clipped)")

plt.tight_layout()
plt.show()

## 13. Additional EDA – Store Type and School Holidays

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=train_df.groupby("StoreType", as_index=False)["Sales"].mean(),
    x="StoreType", y="Sales", ax=axes[0]
)
axes[0].set_title("Average Sales by Store Type")

sns.barplot(
    data=train_df.groupby("SchoolHoliday", as_index=False)["Sales"].mean(),
    x="SchoolHoliday", y="Sales", ax=axes[1]
)
axes[1].set_title("Average Sales by School Holiday")

plt.tight_layout()
plt.show()

# 14. Preprocessing and Feature Engineering

The following function is reusable and creates:
- Year, month, day and quarter
- Week number
- Weekend indicator
- Month start / mid / end indicators
- Cyclical month and day-of-week features
- Competition duration
- Promo2 duration
- Promo interval activity

In [ ]:
def clean_and_engineer(df):
    df = df.copy()

    # Date features
    df["Date"] = pd.to_datetime(df["Date"])
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["Quarter"] = df["Date"].dt.quarter
    df["DayOfWeek"] = df["Date"].dt.dayofweek + 1
    df["IsWeekend"] = df["Date"].dt.dayofweek.isin([5, 6]).astype(int)
    df["IsMonthStart"] = df["Date"].dt.is_month_start.astype(int)
    df["IsMonthEnd"] = df["Date"].dt.is_month_end.astype(int)
    df["IsMonthMid"] = df["Day"].between(11, 20).astype(int)

    # Cyclical features
    df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
    df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)
    df["DOW_sin"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
    df["DOW_cos"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7)

    # State holiday cleanup
    if "StateHoliday" in df.columns:
        df["StateHoliday"] = df["StateHoliday"].astype(str).replace({"0.0": "0"})

    # Competition age
    if {"CompetitionOpenSinceYear", "CompetitionOpenSinceMonth"}.issubset(df.columns):
        comp_year = pd.to_numeric(df["CompetitionOpenSinceYear"], errors="coerce")
        comp_month = pd.to_numeric(df["CompetitionOpenSinceMonth"], errors="coerce")
        df["CompetitionOpenDate"] = pd.to_datetime(
            dict(year=comp_year, month=comp_month, day=1), errors="coerce"
        )
        df["CompetitionOpenMonths"] = (
            (df["Date"].dt.year - df["CompetitionOpenDate"].dt.year) * 12
            + (df["Date"].dt.month - df["CompetitionOpenDate"].dt.month)
        )
        df["CompetitionOpenMonths"] = df["CompetitionOpenMonths"].clip(lower=0)

    # Promo2 age
    if {"Promo2SinceYear", "Promo2SinceWeek"}.issubset(df.columns):
        promo2_start = pd.to_datetime(
            df["Promo2SinceYear"].fillna(2000).astype(int).astype(str)
            + "-W"
            + df["Promo2SinceWeek"].fillna(1).astype(int).astype(str).str.zfill(2)
            + "-1",
            format="%G-W%V-%u",
            errors="coerce"
        )
        df["Promo2Start"] = promo2_start
        df["Promo2RunningWeeks"] = ((df["Date"] - df["Promo2Start"]).dt.days / 7)
        df["Promo2RunningWeeks"] = df["Promo2RunningWeeks"].clip(lower=0)

    # Promo interval activity
    month_abbr = df["Date"].dt.strftime("%b")
    if "PromoInterval" in df.columns:
        interval = df["PromoInterval"].fillna("").astype(str)
        df["IsPromo2Month"] = [
            int((m in x.split(",")) if x else False)
            for m, x in zip(month_abbr, interval)
        ]

    # Drop raw date-construction columns not needed by the final model
    drop_cols = ["CompetitionOpenDate", "Promo2Start"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    return df

full_train = clean_and_engineer(train_df)
full_test = clean_and_engineer(test_df)

print(full_train.shape, full_test.shape)
display(full_train.head())

## 15. Missing Values and Outlier Strategy

In [ ]:
missing_table = full_train.isna().mean().sort_values(ascending=False).to_frame("missing_ratio")
display(missing_table[missing_table["missing_ratio"] > 0])

# For tree-based models, we keep the original target and let the pipeline impute features.
# Extreme target values are not automatically removed because they can represent valid high-sales days.

## 16. Train/Validation Split by Time

In [ ]:
# Exclude closed stores from training because their sales are structurally zero.
model_data = full_train[full_train["Open"] == 1].copy()

# Sort chronologically and use the last 20% as validation.
model_data = model_data.sort_values("Date").reset_index(drop=True)
split_idx = int(len(model_data) * 0.80)

train_model_df = model_data.iloc[:split_idx].copy()
valid_model_df = model_data.iloc[split_idx:].copy()

TARGET = "Sales"

# Production-safe features: Customers is excluded because test data does not contain it.
DROP_FEATURES = [
    "Sales", "Customers", "Date", "Id",
    "PromoInterval"  # handled as IsPromo2Month; raw interval can still be encoded if desired
]

feature_cols = [c for c in model_data.columns if c not in DROP_FEATURES]

X_train = train_model_df[feature_cols]
y_train = train_model_df[TARGET]
X_valid = valid_model_df[feature_cols]
y_valid = valid_model_df[TARGET]

print("Training period:", train_model_df["Date"].min(), "to", train_model_df["Date"].max())
print("Validation period:", valid_model_df["Date"].min(), "to", valid_model_df["Date"].max())
print("Features:", len(feature_cols))

## 17. Build a Scikit-Learn Preprocessing Pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 18. Machine Learning Model – Random Forest Regressor

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=25,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

logger.info("Training Random Forest model")
rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_valid)
rf_pred = np.maximum(rf_pred, 0)

def regression_metrics(y_true, y_pred, name="Model"):
    return pd.DataFrame({
        "Model": [name],
        "MAE": [mean_absolute_error(y_true, y_pred)],
        "RMSE": [mean_squared_error(y_true, y_pred, squared=False)],
        "R2": [r2_score(y_true, y_pred)]
    })

rf_metrics = regression_metrics(y_valid, rf_pred, "Random Forest")
display(rf_metrics)

## 19. Why MAE and RMSE?

- **MAE** is easy to interpret because it gives the average sales error in the original sales units.
- **RMSE** penalizes large forecasting mistakes more strongly, which is useful when large errors can affect inventory and financial planning.
- **R²** is included as an additional explanatory metric.

For this project, **MAE is the primary business-friendly loss metric**, while RMSE is used to monitor large errors.

## 20. Post-Prediction Analysis

In [ ]:
results = valid_model_df[["Date", "Store", "Sales"]].copy()
results["Prediction"] = rf_pred
results["Residual"] = results["Sales"] - results["Prediction"]

display(results.head())

plt.figure(figsize=(8, 6))
plt.scatter(results["Sales"], results["Prediction"], alpha=0.2)
lims = [
    min(results["Sales"].min(), results["Prediction"].min()),
    max(results["Sales"].max(), results["Prediction"].max())
]
plt.plot(lims, lims, "--")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales")
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(results["Residual"], bins=60)
plt.title("Residual Distribution")
plt.show()

## 21. Feature Importance

In [ ]:
# Get transformed feature names
feature_names = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = rf_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

display(importance_df.head(20))

plt.figure(figsize=(10, 8))
sns.barplot(
    data=importance_df.head(20),
    y="Feature", x="Importance"
)
plt.title("Top 20 Feature Importances")
plt.show()

## 22. Prediction Confidence Interval

In [ ]:
# Estimate an empirical 95% prediction interval from validation residuals.
residual_std = results["Residual"].std()
z = 1.96

results["Lower_95"] = np.maximum(0, results["Prediction"] - z * residual_std)
results["Upper_95"] = results["Prediction"] + z * residual_std

coverage = (
    (results["Sales"] >= results["Lower_95"]) &
    (results["Sales"] <= results["Upper_95"])
).mean()

print("Residual standard deviation:", round(residual_std, 2))
print("Approximate empirical coverage:", round(coverage * 100, 2), "%")
display(results.head())

## 23. Serialize the Model with a Timestamp

In [ ]:
timestamp = datetime.now().strftime("%d-%m-%Y-%H-%M-%S")
model_path = MODEL_DIR / f"rossmann_rf_{timestamp}.pkl"

joblib.dump({
    "pipeline": rf_pipeline,
    "feature_columns": feature_cols,
    "metrics": rf_metrics.to_dict(orient="records")[0],
    "created_at": timestamp
}, model_path)

logger.info(f"Model saved to {model_path}")
print("Saved model:", model_path)

## 24. Create Predictions for the Competition Test Data

In [ ]:
X_test = full_test.copy()

# Ensure all required feature columns exist
for col in feature_cols:
    if col not in X_test.columns:
        X_test[col] = np.nan

X_test = X_test[feature_cols]

test_predictions = np.maximum(rf_pipeline.predict(X_test), 0)

# Stores closed on a given day should have zero sales.
test_predictions = np.where(full_test["Open"].fillna(1).eq(0), 0, test_predictions)

submission = pd.DataFrame({
    "Id": test["Id"],
    "Sales": test_predictions
})

submission_path = REPORT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

display(submission.head())
print("Submission saved:", submission_path)

# 25. Time-Series Analysis for Deep Learning

The PDF requires:
1. Isolate time-series data
2. Check stationarity
3. Difference if necessary
4. ACF/PACF
5. Transform to supervised learning using a sliding window
6. Scale data to (-1, 1)
7. Build a two-layer LSTM regression model

In [ ]:
# Aggregate daily sales across stores for a manageable single-series LSTM demonstration.
daily_ts = (
    train_df.groupby("Date")["Sales"]
    .sum()
    .sort_index()
    .asfreq("D")
    .fillna(0)
)

plt.figure(figsize=(16, 5))
plt.plot(daily_ts.index, daily_ts.values)
plt.title("Rossmann Aggregated Daily Sales Time Series")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.show()

print("Observations:", len(daily_ts))

## 26. Stationarity Test – Augmented Dickey-Fuller

In [ ]:
def adf_test(series, name="Series"):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"ADF Test: {name}")
    print("ADF Statistic:", result[0])
    print("p-value:", result[1])
    print("Critical Values:", result[4])
    if result[1] < 0.05:
        print("Conclusion: likely stationary (reject unit-root null hypothesis).")
    else:
        print("Conclusion: likely non-stationary (difference or transform may help).")

adf_test(daily_ts, "Original Daily Sales")

ts_diff = daily_ts.diff().dropna()
adf_test(ts_diff, "First-Differenced Daily Sales")

## 27. ACF and PACF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_acf(ts_diff, lags=min(40, len(ts_diff)//2 - 1), ax=axes[0])
plot_pacf(ts_diff, lags=min(40, len(ts_diff)//2 - 1), ax=axes[1], method="ywm")
plt.tight_layout()
plt.show()

## 28. Sliding Window Transformation and Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Use the original series for LSTM. A log transform can stabilize very large values.
ts_values = np.log1p(daily_ts.values.reshape(-1, 1))

lstm_scaler = MinMaxScaler(feature_range=(-1, 1))
scaled_ts = lstm_scaler.fit_transform(ts_values)

def create_sequences(values, window_size=14):
    X, y = [], []
    for i in range(window_size, len(values)):
        X.append(values[i-window_size:i, 0])
        y.append(values[i, 0])
    return np.array(X), np.array(y)

WINDOW_SIZE = 14
X_seq, y_seq = create_sequences(scaled_ts, WINDOW_SIZE)

X_seq = X_seq.reshape((X_seq.shape[0], X_seq.shape[1], 1))

split = int(len(X_seq) * 0.80)
X_seq_train, X_seq_valid = X_seq[:split], X_seq[split:]
y_seq_train, y_seq_valid = y_seq[:split], y_seq[split:]

print("X train:", X_seq_train.shape)
print("X validation:", X_seq_valid.shape)

## 29. LSTM Deep Learning Model

In [ ]:
# TensorFlow is optional because some local environments may not have it installed.
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping

    tf.random.set_seed(RANDOM_STATE)

    lstm_model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
        LSTM(32),
        Dense(16, activation="relu"),
        Dense(1)
    ])

    lstm_model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    history = lstm_model.fit(
        X_seq_train, y_seq_train,
        validation_data=(X_seq_valid, y_seq_valid),
        epochs=50,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1
    )

    print(lstm_model.summary())

except ImportError:
    print("TensorFlow is not installed. Run: pip install tensorflow")

## 30. Evaluate the LSTM Model

In [ ]:
try:
    lstm_pred_scaled = lstm_model.predict(X_seq_valid)

    y_true_log = lstm_scaler.inverse_transform(y_seq_valid.reshape(-1, 1))
    y_pred_log = lstm_scaler.inverse_transform(lstm_pred_scaled)

    y_true_sales = np.expm1(y_true_log.ravel())
    y_pred_sales = np.maximum(np.expm1(y_pred_log.ravel()), 0)

    lstm_metrics = regression_metrics(
        y_true_sales, y_pred_sales, "LSTM"
    )
    display(lstm_metrics)

    plt.figure(figsize=(12, 5))
    plt.plot(y_true_sales, label="Actual")
    plt.plot(y_pred_sales, label="Predicted")
    plt.title("LSTM: Actual vs Predicted Aggregated Daily Sales")
    plt.legend()
    plt.show()

    lstm_path = MODEL_DIR / f"rossmann_lstm_{timestamp}.keras"
    lstm_model.save(lstm_path)
    print("LSTM model saved:", lstm_path)

except NameError:
    print("LSTM evaluation skipped because TensorFlow/model is unavailable.")

## 31. MLflow Experiment Tracking

In [ ]:
# This section logs model parameters and metrics if MLflow is installed.
try:
    import mlflow
    import mlflow.sklearn

    mlflow.set_experiment("Rossmann_Store_Sales")

    with mlflow.start_run(run_name=f"RandomForest_{timestamp}"):
        mlflow.log_params({
            "model": "RandomForestRegressor",
            "n_estimators": 200,
            "max_depth": 25,
            "random_state": RANDOM_STATE
        })

        metrics_dict = rf_metrics.iloc[0].drop("Model").to_dict()
        mlflow.log_metrics({k: float(v) for k, v in metrics_dict.items()})

        mlflow.sklearn.log_model(
            rf_pipeline,
            name="rossmann_random_forest"
        )

    print("MLflow run completed.")
    print("To open MLflow UI, run in terminal:")
    print("mlflow ui --port 5000")

except ImportError:
    print("MLflow is not installed. Run: pip install mlflow")
except Exception as e:
    print("MLflow section skipped:", e)

## 32. Final Findings and Business Recommendations

### Main analytical conclusions to update after running the notebook
- Promotions should be evaluated by **incremental sales and customer lift**, not only raw sales.
- Holidays and seasonal periods can change purchasing patterns substantially.
- Customer count has a direct relationship with sales and is valuable for analysis, but should not be used as a future production feature unless it is separately forecast.
- Store type, assortment, competition distance and promotion history can help explain differences between stores.
- A time-based validation split is preferable to a random split for future sales forecasting.
- Random Forest provides a strong baseline for tabular store-level prediction.
- LSTM provides a separate deep-learning approach for sequential sales patterns.
- Prediction intervals should be communicated to managers because forecasts contain uncertainty.

## 33. Submission Checklist

- [x] Data loading and merging
- [x] Missing-value and duplicate checks
- [x] Promotion distribution comparison
- [x] Holiday analysis
- [x] Seasonal analysis
- [x] Sales and customers correlation
- [x] Promotion impact analysis
- [x] Open/closed and weekend behaviour
- [x] Assortment and competition analysis
- [x] Python logging
- [x] Date and business feature engineering
- [x] Scikit-learn preprocessing pipeline
- [x] Random Forest regression model
- [x] MAE, RMSE and R² evaluation
- [x] Feature importance
- [x] Prediction confidence interval
- [x] Timestamped model serialization
- [x] Test-set submission generation
- [x] Stationarity testing
- [x] ACF/PACF
- [x] Sliding-window supervised learning
- [x] Scaling to (-1, 1)
- [x] Two-layer LSTM
- [x] MLflow integration

### Separate project files still needed for a complete repository
- `app.py` / Streamlit dashboard
- `README.md`
- `requirements.txt`
- `Dockerfile`
- `.github/workflows/ci.yml`
- `tests/`
- presentation slides
- final PDF report